# SCFP V2 diagnostic audit — không retrain
Attach checkpoint V2 đã train và ICFG-PEDES, bật GPU + Internet. Ba cell: cấu hình, source/dependencies, audit.

Dùng SOURCE_ZIP để chạy mã local chưa push; tạo ZIP bằng `python -m experiments.scfp.bundle --output IRRA_scfp_v2_audit_source.zip`, rồi attach vào Kaggle. Nếu dùng GitHub, branch phải có audit.py mới. Notebook không tự push.

Mặc định đo 128 cặp train (32 PID × tối đa 4 ảnh), 6 cặp minh họa chọn trước theo độ dài caption và đa dạng PID; ba batch gradient, mỗi batch 8 cặp. R1 ON/OFF là tập con train đã được mô hình thấy, không phải official test hay validation độc lập. Không optimizer, không lưu trọng số, không tuning bằng test.


In [ ]:
from pathlib import Path
SOURCE_ZIP = None  # /kaggle/input/.../IRRA_scfp_v2_audit_source.zip; tự tìm nếu duy nhất
CHECKPOINT = None  # /kaggle/input/.../best.pth V2; tự tìm nếu chỉ có một best.pth
CONFIG_FILE = None  # configs.yaml đúng run; có thể bỏ nếu checkpoint nhúng config
ANNOTATION_SOURCE = None  # chỉ đặt nếu có nhiều ICFG-PEDES.json
IMAGES_SOURCE = None  # thư mục chứa các đường dẫn ảnh trong annotation
INPUT_ROOT = "/kaggle/input"
REPO_URL = "https://github.com/thanhphu25/IRRA.git"
BRANCH = "feat/scfp-v2"
PRECISION = "checkpoint"  # fp32 để audit riêng nếu cần phân biệt gradient rất nhỏ với underflow
SEED = 2026
NUM_IDENTITIES = 32
IMAGES_PER_IDENTITY = 4
BATCH_SIZE = 16
GRADIENT_BATCH_SIZE = 8
GRADIENT_BATCHES = 3
EXAMPLES = 6


In [ ]:
import os, sys, subprocess, tempfile, zipfile, hashlib, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if SOURCE_ZIP is None:
    found = sorted(Path(INPUT_ROOT).rglob("IRRA_scfp_v2_audit_source.zip"))
    if len(found) == 1:
        SOURCE_ZIP = str(found[0])
    elif len(found) > 1:
        raise ValueError(f"Chọn SOURCE_ZIP cụ thể: {found}")
REPO = Path(tempfile.mkdtemp(prefix="IRRA_scfp_v2_audit_", dir="/kaggle/working")).resolve()
if SOURCE_ZIP:
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for info in archive.infolist():
            if not (REPO / info.filename).resolve().is_relative_to(REPO):
                raise ValueError(f"Unsafe ZIP path: {info.filename}")
        archive.extractall(REPO)
    manifest_path = REPO / "source_manifest.json"
    if manifest_path.is_file():
        for name, expected in json.loads(manifest_path.read_text()).items():
            if not (REPO / name).resolve().is_relative_to(REPO):
                raise ValueError(f"Unsafe manifest path: {name}")
            assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
else:
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
if not (REPO / "experiments/scfp/audit.py").is_file():
    raise RuntimeError("Source chưa có audit mới. Attach SOURCE_ZIP được tạo từ workspace đã cập nhật.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "experiments/scfp/requirements-audit.txt")], check=True)
import torch
assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings"
print("Source:", REPO, "| torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))


In [ ]:
if CHECKPOINT is None:
    candidates = sorted(Path(INPUT_ROOT).rglob("best.pth"))
    if len(candidates) != 1:
        raise ValueError(f"Đặt CHECKPOINT là best.pth V2 chính xác. Các lựa chọn: {candidates}")
    CHECKPOINT = str(candidates[0])
# Mỗi lần chạy tạo thư mục mới, không ghi đè kết quả cũ.
RUN_ROOT = Path(tempfile.mkdtemp(prefix="SCFP_V2_AUDIT_", dir="/kaggle/working"))
DATA_ROOT = RUN_ROOT / "data"
AUDIT_OUTPUT = RUN_ROOT / "results"
prepare = [sys.executable, "-m", "experiments.scfp.prepare_kaggle", "--input_root", INPUT_ROOT, "--data_root", str(DATA_ROOT)]
if ANNOTATION_SOURCE:
    prepare += ["--annotation_source", ANNOTATION_SOURCE]
if IMAGES_SOURCE:
    prepare += ["--images_source", IMAGES_SOURCE]
subprocess.run(prepare, cwd=REPO, check=True)
command = [sys.executable, "-m", "experiments.scfp.audit", "--checkpoint", CHECKPOINT,
           "--root-dir", str(DATA_ROOT), "--output-dir", str(AUDIT_OUTPUT), "--device", "cuda",
           "--precision", PRECISION, "--seed", str(SEED), "--num-identities", str(NUM_IDENTITIES),
           "--images-per-identity", str(IMAGES_PER_IDENTITY), "--batch-size", str(BATCH_SIZE),
           "--gradient-batch-size", str(GRADIENT_BATCH_SIZE), "--gradient-batches", str(GRADIENT_BATCHES),
           "--examples", str(EXAMPLES)]
if CONFIG_FILE:
    command += ["--config-file", CONFIG_FILE]
completed = subprocess.run(command, cwd=REPO)
if completed.returncode not in (0, 2) or not (AUDIT_OUTPUT / "SCFP_V2_AUDIT.md").is_file():
    raise RuntimeError(f"Audit thất bại: exit {completed.returncode}; xem log phía trên")
from IPython.display import display, Markdown, Image
print("Outputs:", AUDIT_OUTPUT, "| exit:", completed.returncode, "(2 = có phép đo unavailable/partial)")
display(Markdown((AUDIT_OUTPUT / "SCFP_V2_AUDIT.md").read_text()))
for name in ("frequency_response.png", "visual_slot_attention.png", "text_slot_attention.png"):
    display(Image(filename=str(AUDIT_OUTPUT / name)))
